# 电商平台重点商家经营异常诊断

**决策问题：** 目标商家的经营下滑是否显著，哪些可观测因素值得优先核查，平台应采取什么行动？

**分析原则：**

1. 使用完整月份，避免数据尾部造成虚假下滑；
2. 使用平台、主营品类和同类商家三层基准；
3. 区分数据事实、业务解释和待验证假设；
4. 不虚构库存、曝光、利润或实验结果。

In [1]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

WORKSPACE = Path.cwd()
if (WORKSPACE / "project").exists():
    PROJECT = WORKSPACE / "project"
elif (WORKSPACE.parent / "data" / "raw").exists():
    PROJECT = WORKSPACE.parent
else:
    PROJECT = WORKSPACE
ZIP_PATH = PROJECT / "data" / "raw" / "olist_brazilian_ecommerce.zip"
OUT = PROJECT / "data" / "results"
ASSETS = PROJECT / "analysis" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

TARGET_SELLER = "da8622b14eb17ae2831f4ac5b9dab84a"
TARGET_LABEL = "S-DA862"
START_DATE = "2017-01-01"
END_DATE = "2018-08-01"  # 只使用截至2018年7月的完整月份
BASE_MONTH = pd.Period("2018-04")
END_MONTH = pd.Period("2018-07")
FOCUS_MONTHS = pd.period_range(BASE_MONTH, END_MONTH, freq="M")

RED = "#C83B22"
BLUE = "#2878B5"
GREY = "#81909C"
GREEN = "#2F7D5B"
ORANGE = "#D8782D"

def font(size=24, bold=False):
    return ImageFont.load_default(size=size)

def line_chart(series, periods, title, path, colors):
    width, height = 1200, 520
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    left, top, right, bottom = 105, 75, 45, 85
    pw, ph = width-left-right, height-top-bottom
    values = [v for vals in series.values() for v in vals]
    ymin = min(0, math.floor(min(values)/20)*20)
    ymax = max(120, math.ceil(max(values)/20)*20)
    draw.text((left, 20), title, font=font(28, True), fill="#1F2933")
    for tick in np.linspace(ymin, ymax, 6):
        y = top + ph - (tick-ymin)/(ymax-ymin)*ph
        draw.line((left, y, width-right, y), fill="#E5E8EB", width=1)
        draw.text((20, y-12), f"{tick:.0f}", font=font(17), fill="#5D6872")
    xs = np.linspace(left, width-right, len(periods))
    for x, period in zip(xs, periods):
        draw.text((x-35, height-bottom+18), period, font=font(17), fill="#5D6872")
    for label, vals in series.items():
        pts = []
        for x, value in zip(xs, vals):
            y = top + ph - (value-ymin)/(ymax-ymin)*ph
            pts.append((x, y))
        draw.line(pts, fill=colors[label], width=5)
        for x, y in pts:
            draw.ellipse((x-6, y-6, x+6, y+6), fill=colors[label])
        x, y = pts[-1]
        draw.text((x-25, y-34), f"{vals[-1]:.1f}", font=font(18, True), fill=colors[label])
    lx = left
    for label in series:
        draw.line((lx, height-28, lx+32, height-28), fill=colors[label], width=5)
        draw.text((lx+42, height-42), label, font=font(17), fill="#34404A")
        lx += 190
    img.save(path)

def vertical_bar(labels, values, title, path, colors, value_format="{:.0f}", ylabel=""):
    width, height = 1100, 500
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    left, top, right, bottom = 120, 75, 50, 105
    pw, ph = width-left-right, height-top-bottom
    vmin = min(0, min(values)*1.18)
    vmax = max(0, max(values)*1.18)
    if vmax == vmin:
        vmax = vmin + 1
    zero_y = top + ph - (0-vmin)/(vmax-vmin)*ph
    draw.text((left, 20), title, font=font(28, True), fill="#1F2933")
    draw.line((left, zero_y, width-right, zero_y), fill="#7A8793", width=2)
    step = pw/len(labels)
    bw = step*0.48
    for idx, (label, value) in enumerate(zip(labels, values)):
        x = left + step*(idx+.5)
        y = top + ph - (value-vmin)/(vmax-vmin)*ph
        draw.rectangle((x-bw/2, min(y, zero_y), x+bw/2, max(y, zero_y)), fill=colors[idx])
        label_box = draw.textbbox((0, 0), label, font=font(18))
        draw.text((x-(label_box[2]-label_box[0])/2, height-bottom+25), label, font=font(18), fill="#34404A")
        value_text = value_format.format(value)
        value_box = draw.textbbox((0, 0), value_text, font=font(20, True))
        ty = y-32 if value >= 0 else y+8
        draw.text((x-(value_box[2]-value_box[0])/2, ty), value_text, font=font(20, True), fill="#1F2933")
    if ylabel:
        draw.text((20, top+ph/2), ylabel, font=font(17), fill="#5D6872")
    img.save(path)

def peer_dot_chart(peer_frame, target_id, median, q1, q3, path):
    width, height = 1200, 520
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    left, top, right, bottom = 130, 65, 70, 80
    pw, ph = width-left-right, height-top-bottom
    frame = peer_frame.sort_values("gmv_change").reset_index(drop=True)
    raw = frame["gmv_change"].mul(100)
    xmin = math.floor(min(-60, raw.min())/20)*20
    xmax = 120
    draw.text((left, 18), "同类商家GMV变化分布（2018-04至2018-07）", font=font(28, True), fill="#1F2933")
    for tick in range(int(xmin), xmax+1, 20):
        x = left + (tick-xmin)/(xmax-xmin)*pw
        draw.line((x, top, x, top+ph), fill="#E5E8EB", width=1)
        draw.text((x-20, height-bottom+18), f"{tick}%", font=font(16), fill="#5D6872")
    x0 = left + (0-xmin)/(xmax-xmin)*pw
    draw.line((x0, top, x0, top+ph), fill="#7A8793", width=2)
    ys = np.linspace(top+12, top+ph-12, len(frame))
    for y, (_, row) in zip(ys, frame.iterrows()):
        value = row["gmv_change"]*100
        shown = min(value, xmax)
        x = left + (shown-xmin)/(xmax-xmin)*pw
        color = RED if row["seller_id"] == target_id else "#8FA6B8"
        radius = 9 if row["seller_id"] == target_id else 6
        draw.ellipse((x-radius, y-radius, x+radius, y+radius), fill=color)
        if row["seller_id"] == target_id:
            draw.text((x+14, y-13), f"S-DA862  {value:.1f}%", font=font(18, True), fill=RED)
        elif value > xmax:
            draw.text((x-5, y-12), "→", font=font(18, True), fill="#5D6872")
            draw.text((x+18, y-12), f"{value:.0f}%", font=font(16), fill="#5D6872")
    band_x1 = left + (q1*100-xmin)/(xmax-xmin)*pw
    band_x2 = left + (q3*100-xmin)/(xmax-xmin)*pw
    med_x = left + (median*100-xmin)/(xmax-xmin)*pw
    draw.line((band_x1, top+ph+12, band_x2, top+ph+12), fill=BLUE, width=7)
    draw.line((med_x, top+ph+3, med_x, top+ph+21), fill=BLUE, width=3)
    draw.text((band_x1, top+ph+28), f"同类IQR  {q1:.1%} 至 {q3:.1%}｜中位数 {median:.1%}", font=font(17), fill=BLUE)
    img.save(path)

print(f"数据文件：{ZIP_PATH.name}")
print(f"主分析期：{BASE_MONTH} 至 {END_MONTH}（完整月份）")

数据文件：olist_brazilian_ecommerce.zip
主分析期：2018-04 至 2018-07（完整月份）


## 1. 数据加载与口径

- GMV：已交付订单商品金额 `price` 之和，不含运费，货币单位按原始数据记为 BRL；
- 订单量：已交付订单去重数；客单价：GMV / 订单量；
- 主分析期：2018年4月至7月。原始数据在2018年8月末明显进入采集尾部，因此8月不进入核心结论；
- “成交SKU”只表示当月有成交记录，不能直接解释为商品上下架或库存状态。

In [2]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    orders = pd.read_csv(
        zf.open("olist_orders_dataset.csv"),
        parse_dates=[
            "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ],
    )
    items = pd.read_csv(zf.open("olist_order_items_dataset.csv"))
    products = pd.read_csv(zf.open("olist_products_dataset.csv"))
    translations = pd.read_csv(zf.open("product_category_name_translation.csv"))
    reviews = pd.read_csv(zf.open("olist_order_reviews_dataset.csv"))

products = products.merge(translations, on="product_category_name", how="left")
reviews = reviews.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")
fact = (
    items
    .merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .merge(orders, on="order_id", how="inner")
    .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
)
fact = fact[
    (fact["order_status"] == "delivered")
    & (fact["order_purchase_timestamp"] >= START_DATE)
    & (fact["order_purchase_timestamp"] < END_DATE)
].copy()
fact["month"] = fact["order_purchase_timestamp"].dt.to_period("M")

august_all = orders[orders["order_purchase_timestamp"].dt.to_period("M") == pd.Period("2018-08")]
august_daily = august_all.groupby(august_all["order_purchase_timestamp"].dt.day).size()
quality = {
    "delivered_orders": int(fact["order_id"].nunique()),
    "sellers": int(fact["seller_id"].nunique()),
    "review_coverage": float(fact.drop_duplicates("order_id")["review_score"].notna().mean()),
    "august_last_three_days_orders": int(august_daily.reindex([29, 30, 31], fill_value=0).sum()),
}
print(pd.Series(quality).to_string())

delivered_orders                 89860.000000
sellers                           2793.000000
review_coverage                      0.993078
august_last_three_days_orders       19.000000


## 2. 候选商家筛选

规则要求历史规模足够、2018年4月至7月持续活跃、GMV连续下降且7月仍有一定订单量。筛选规则先于结论确定，避免先挑故事再找数据。

In [3]:
seller_month = (
    fact.groupby(["seller_id", "month"])
    .agg(orders=("order_id", "nunique"), gmv=("price", "sum"), active_skus=("product_id", "nunique"))
    .reset_index()
    .sort_values(["seller_id", "month"])
)
seller_summary = seller_month.groupby("seller_id").agg(
    total_orders=("orders", "sum"), active_months=("month", "nunique")
)
pivot = seller_month[seller_month["month"].isin(FOCUS_MONTHS)].pivot(
    index="seller_id", columns="month", values=["orders", "gmv"]
).dropna()
pivot.columns = [f"{metric}_{period}" for metric, period in pivot.columns]
pivot = pivot.join(seller_summary)
eligible = pivot[
    (pivot["total_orders"] >= 100)
    & (pivot["active_months"] >= 6)
    & (pivot["orders_2018-04"] >= 50)
    & (pivot["orders_2018-07"] >= 20)
].copy()
eligible["continuous_decline"] = eligible.apply(
    lambda r: r["gmv_2018-04"] > r["gmv_2018-05"] > r["gmv_2018-06"] > r["gmv_2018-07"], axis=1
)
eligible["gmv_drop"] = eligible["gmv_2018-07"] / eligible["gmv_2018-04"] - 1
eligible["gmv_loss"] = eligible["gmv_2018-04"] - eligible["gmv_2018-07"]
candidate_table = eligible[eligible["continuous_decline"]].sort_values("gmv_loss", ascending=False)
print(f"候选商家：{len(candidate_table)} 家")
print(candidate_table[["total_orders", "gmv_drop", "gmv_loss"]].head(8).round(3).to_string())
print(f"目标商家进入候选池：{TARGET_SELLER in candidate_table.index}")

候选商家：4 家
                                  total_orders  gmv_drop  gmv_loss
seller_id                                                         
da8622b14eb17ae2831f4ac5b9dab84a          1253    -0.521   7288.49
1f50f920176fa81dab994f9023523100          1364    -0.725   6135.34
7a67c85e85bb2ce8582c35f2203ad736          1128    -0.602   3722.12
8b321bb669392f5163d04c59e235e066           922    -0.686   1401.59
目标商家进入候选池：True


## 3. 异常确认：平台、品类与同类商家三层基准

平台和主营品类用于排除共同波动；同品类且首尾月份均有至少5单的商家用于位置比较。中位数和四分位区间比单一排名更稳健。

In [4]:
target = fact[fact["seller_id"] == TARGET_SELLER].copy()
category_gmv = target.groupby("product_category_name_english")["price"].sum().sort_values(ascending=False)
MAIN_CATEGORY = category_gmv.index[0]
main_category_share = float(category_gmv.iloc[0] / category_gmv.sum())

def monthly_metrics(df):
    result = df.groupby("month").agg(orders=("order_id", "nunique"), gmv=("price", "sum")).reset_index()
    result["aov"] = result["gmv"] / result["orders"]
    return result

target_monthly = monthly_metrics(target)
category_monthly = monthly_metrics(fact[fact["product_category_name_english"] == MAIN_CATEGORY])
platform_monthly = monthly_metrics(fact)

comparison_rows = []
for label, frame in [(TARGET_LABEL, target_monthly), ("主营品类", category_monthly), ("平台", platform_monthly)]:
    part = frame[frame["month"].isin(FOCUS_MONTHS)].copy()
    part["gmv_index"] = part["gmv"] / part.iloc[0]["gmv"] * 100
    part["orders_index"] = part["orders"] / part.iloc[0]["orders"] * 100
    part["series"] = label
    comparison_rows.append(part)
comparison = pd.concat(comparison_rows, ignore_index=True)

category_seller_month = (
    fact[fact["product_category_name_english"] == MAIN_CATEGORY]
    .groupby(["seller_id", "month"])
    .agg(orders=("order_id", "nunique"), gmv=("price", "sum"))
    .reset_index()
)
peer = category_seller_month[category_seller_month["month"].isin([BASE_MONTH, END_MONTH])].pivot(
    index="seller_id", columns="month", values=["orders", "gmv"]
).dropna()
peer.columns = [f"{metric}_{period}" for metric, period in peer.columns]
peer = peer[(peer["orders_2018-04"] >= 5) & (peer["orders_2018-07"] >= 5)].copy()
peer["gmv_change"] = peer["gmv_2018-07"] / peer["gmv_2018-04"] - 1
peer_reset = peer.reset_index()
target_category_change = float(peer.loc[TARGET_SELLER, "gmv_change"])
peer_median_change = float(peer["gmv_change"].median())
peer_q1 = float(peer["gmv_change"].quantile(.25))
peer_q3 = float(peer["gmv_change"].quantile(.75))
target_rank = int(peer["gmv_change"].rank(method="min", ascending=True).loc[TARGET_SELLER])

colors = {TARGET_LABEL: RED, "主营品类": BLUE, "平台": GREY}
labels = [str(p) for p in FOCUS_MONTHS]
line_chart(
    {label: comparison[comparison["series"] == label]["gmv_index"].tolist() for label in colors},
    labels, "GMV指数（2018-04=100）", ASSETS / "gmv_trend.png", colors,
)
line_chart(
    {label: comparison[comparison["series"] == label]["orders_index"].tolist() for label in colors},
    labels, "订单量指数（2018-04=100）", ASSETS / "orders_trend.png", colors,
)
peer_dot_chart(peer_reset, TARGET_SELLER, peer_median_change, peer_q1, peer_q3, ASSETS / "peer_benchmark.png")

print(f"主营品类：{MAIN_CATEGORY}，历史GMV占比：{main_category_share:.1%}")
print(f"同类商家：{len(peer)} 家；中位数：{peer_median_change:.1%}；IQR：{peer_q1:.1%} 至 {peer_q3:.1%}")
print(f"目标商家主营品类变化：{target_category_change:.1%}；降幅排名：第 {target_rank}")

主营品类：bed_bath_table，历史GMV占比：95.4%
同类商家：15 家；中位数：-17.6%；IQR：-40.6% 至 37.9%
目标商家主营品类变化：-54.0%；降幅排名：第 3


## 4. GMV驱动拆解

使用恒等式 `GMV = 订单量 × 客单价` 和中点分解。该方法量化下降表现，不把恒等式误写为因果证明。

In [5]:
target_focus = target_monthly[target_monthly["month"].isin(FOCUS_MONTHS)].set_index("month")
base = target_focus.loc[BASE_MONTH]
end = target_focus.loc[END_MONTH]
delta_gmv = float(end["gmv"] - base["gmv"])
order_effect = float((end["orders"] - base["orders"]) * (base["aov"] + end["aov"]) / 2)
aov_effect = float((end["aov"] - base["aov"]) * (base["orders"] + end["orders"]) / 2)
driver_summary = pd.DataFrame({
    "指标": ["GMV（BRL）", "订单量", "客单价（BRL）"],
    "2018-04": [base["gmv"], base["orders"], base["aov"]],
    "2018-07": [end["gmv"], end["orders"], end["aov"]],
})
driver_summary["变化率"] = driver_summary["2018-07"] / driver_summary["2018-04"] - 1
print(driver_summary.round(2).to_string(index=False))
print(f"订单量效应：{order_effect:,.0f} BRL")
print(f"客单价效应：{aov_effect:,.0f} BRL")
vertical_bar(
    ["订单量效应", "客单价效应"], [order_effect, aov_effect],
    "GMV下降的中点分解", ASSETS / "gmv_driver_decomposition.png",
    [BLUE, ORANGE], "{:,.0f}", "BRL",
)

      指标  2018-04  2018-07   变化率
GMV（BRL） 13995.77  6707.28 -0.52
     订单量   118.00    65.00 -0.45
客单价（BRL）   118.61   103.19 -0.13
订单量效应：-5,878 BRL
客单价效应：-1,411 BRL


## 5. 成交SKU迁移与交易结构

将4月和7月成交SKU分成“持续成交、仅4月成交、7月新增成交”。这可以定位优先核查对象，但“未成交”不等于缺货或下架。

In [6]:
base_items = target[target["month"] == BASE_MONTH].copy()
end_items = target[target["month"] == END_MONTH].copy()
base_skus = set(base_items["product_id"])
end_skus = set(end_items["product_id"])
retained_skus = base_skus & end_skus
base_only_skus = base_skus - end_skus
end_new_skus = end_skus - base_skus

retained_base_gmv = float(base_items[base_items["product_id"].isin(retained_skus)]["price"].sum())
retained_end_gmv = float(end_items[end_items["product_id"].isin(retained_skus)]["price"].sum())
base_only_gmv = float(base_items[base_items["product_id"].isin(base_only_skus)]["price"].sum())
end_new_gmv = float(end_items[end_items["product_id"].isin(end_new_skus)]["price"].sum())
sku_effects = {
    "仅4月成交SKU": -base_only_gmv,
    "持续成交SKU变化": retained_end_gmv-retained_base_gmv,
    "7月新增成交SKU": end_new_gmv,
}
sku_summary = {
    "base_active_skus": len(base_skus),
    "end_active_skus": len(end_skus),
    "retained_skus": len(retained_skus),
    "base_only_skus": len(base_only_skus),
    "end_new_skus": len(end_new_skus),
    "sku_retention": len(retained_skus)/len(base_skus),
    "base_only_gmv_share": base_only_gmv/float(base["gmv"]),
    "retained_gmv_change": retained_end_gmv/retained_base_gmv-1,
    "sku_per_100_orders_base": len(base_skus)/float(base["orders"])*100,
    "sku_per_100_orders_end": len(end_skus)/float(end["orders"])*100,
    "item_price_median_base": float(base_items["price"].median()),
    "item_price_median_end": float(end_items["price"].median()),
    "active_sku_price_median_base": float(base_items.groupby("product_id")["price"].median().median()),
    "active_sku_price_median_end": float(end_items.groupby("product_id")["price"].median().median()),
}
print(pd.Series(sku_summary).round(3).to_string())
print("SKU迁移对GMV变化的描述性拆解：")
print(pd.Series(sku_effects).round(2).to_string())

vertical_bar(
    list(sku_effects), list(sku_effects.values()),
    "成交SKU迁移对应的GMV变化", ASSETS / "sku_migration.png",
    [RED, ORANGE, GREEN], "{:,.0f}", "BRL",
)
vertical_bar(
    ["4月单件成交", "7月单件成交", "4月成交SKU", "7月成交SKU"],
    [sku_summary["item_price_median_base"], sku_summary["item_price_median_end"],
     sku_summary["active_sku_price_median_base"], sku_summary["active_sku_price_median_end"]],
    "成交价格中位数：交易组合下移，但成交SKU中位价稳定", ASSETS / "transaction_price_mix.png",
    [GREY, RED, BLUE, BLUE], "{:.1f}", "BRL",
)

base_active_skus                 63.000
end_active_skus                  37.000
retained_skus                    26.000
base_only_skus                   37.000
end_new_skus                     11.000
sku_retention                     0.413
base_only_gmv_share               0.550
retained_gmv_change              -0.212
sku_per_100_orders_base          53.390
sku_per_100_orders_end           56.923
item_price_median_base           99.900
item_price_median_end            44.900
active_sku_price_median_base    109.900
active_sku_price_median_end     109.900
SKU迁移对GMV变化的描述性拆解：
仅4月成交SKU    -7694.20
持续成交SKU变化   -1337.69
7月新增成交SKU    1743.40


## 6. 结论、行动与验证

| 层级 | 结论 |
|---|---|
| 数据事实 | 4月至7月GMV连续下降，降幅显著超过平台、主营品类和多数同类商家；订单量是最大量化驱动。 |
| 结构信号 | 4月仅当月成交的37个SKU贡献了当月55.0%的GMV；7月新增成交SKU无法弥补缺口。 |
| 克制解释 | 交易商品组合向较低金额迁移，但成交SKU中位价保持稳定，不能直接写成商家主动降价。 |
| 待验证假设 | 历史高贡献SKU可能因库存、上下架、曝光、价格竞争力或需求变化而未再成交。 |

**建议：**

1. **P0 高贡献SKU核查：** 核查4月高贡献但7月未成交SKU的库存、上下架、曝光和竞争价格；
2. **P1 SKU恢复或替代试点：** 选择可恢复SKU或同价带替代品，与匹配SKU/商家进行4周对照；
3. **P2 重点商家异常监控：** 建立连续下滑、同类偏离和高贡献SKU流失预警。

**成功指标：** 增量GMV、订单量、客单价；**护栏指标：** 毛利率、取消率、退款率。没有利润和实验数据时，不声称方案已经产生提升。

In [7]:
platform_focus = platform_monthly.set_index("month").loc[FOCUS_MONTHS]
category_focus = category_monthly.set_index("month").loc[FOCUS_MONTHS]
summary = {
    "target_seller": TARGET_SELLER,
    "target_label": TARGET_LABEL,
    "analysis_start": str(BASE_MONTH),
    "analysis_end": str(END_MONTH),
    "main_category": MAIN_CATEGORY,
    "main_category_share": main_category_share,
    "total_target_orders": int(target["order_id"].nunique()),
    "base_gmv": float(base["gmv"]),
    "end_gmv": float(end["gmv"]),
    "gmv_change": float(end["gmv"] / base["gmv"] - 1),
    "base_orders": int(base["orders"]),
    "end_orders": int(end["orders"]),
    "orders_change": float(end["orders"] / base["orders"] - 1),
    "base_aov": float(base["aov"]),
    "end_aov": float(end["aov"]),
    "aov_change": float(end["aov"] / base["aov"] - 1),
    "platform_change": float(platform_focus.iloc[-1]["gmv"] / platform_focus.iloc[0]["gmv"] - 1),
    "category_change": float(category_focus.iloc[-1]["gmv"] / category_focus.iloc[0]["gmv"] - 1),
    "order_effect": order_effect,
    "aov_effect": aov_effect,
    "peer_count": int(len(peer)),
    "peer_median_change": peer_median_change,
    "peer_q1": peer_q1,
    "peer_q3": peer_q3,
    "target_category_change": target_category_change,
    "target_rank": target_rank,
    "quality": quality,
    "sku": sku_summary,
    "sku_effects": sku_effects,
}
(OUT / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
target_monthly.to_csv(OUT / "merchant_monthly_metrics.csv", index=False, encoding="utf-8-sig")
peer_reset.to_csv(OUT / "peer_benchmark.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([
    {"sku_group": "retained", "sku_count": len(retained_skus), "base_gmv": retained_base_gmv, "end_gmv": retained_end_gmv},
    {"sku_group": "base_only", "sku_count": len(base_only_skus), "base_gmv": base_only_gmv, "end_gmv": 0},
    {"sku_group": "end_new", "sku_count": len(end_new_skus), "base_gmv": 0, "end_gmv": end_new_gmv},
]).to_csv(OUT / "sku_migration.csv", index=False, encoding="utf-8-sig")
print("已写出分析摘要、月度指标、同类商家和SKU迁移结果。")
print(f"核心结论：GMV {summary['gmv_change']:.1%}；订单量 {summary['orders_change']:.1%}；客单价 {summary['aov_change']:.1%}")

已写出分析摘要、月度指标、同类商家和SKU迁移结果。
核心结论：GMV -52.1%；订单量 -44.9%；客单价 -13.0%
